# Fiaxell SOFC run: from AMBER to results (DEMO)

**The data in this notebook are fictitious.** They were generated by `demo_data/make_demo_data.py` in the layout the
Fiaxell station program writes, so the whole path can be shown before real data exist.

What the notebook shows, in order:

1. **AMBER** – the run as AMBER shows it to you (your own read-only key; nothing is written to AMBER).
2. **The data folder** – `AMBER_run.json` says which run, experiment and samples the files belong to.
3. **What happened** – furnace temperature and gas flows over the whole run.
4. **OCV check** – measured open-circuit voltage against the Nernst value.
5. **I-V** – voltage and power density; peak power and area specific resistance (ASR).
6. **EIS** – Nyquist plots; ohmic and polarisation resistance.
7. **Arrhenius** – activation energies from the three temperatures.
8. **Result** – a short table to copy into the experiment in AMBER.

Data that are not confidential: Colab is fine. Confidential data: the group's Jupyter or your own computer.

## 0. Setup

Installs the AMBER client from the group's GitHub (a few seconds in Colab or Jupyter).

In [ ]:
%pip install -q git+https://github.com/dfme-lab/amber-notebooks

In [ ]:
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from amber_client import Amber, AmberError, load_run_json

RUN = "ST05-26r001"                       # the run code from the run page in AMBER
AMBER_URL = "https://test.next.dfme.eu"   # the test instance; the real one is https://amber.dfme.eu
FOLDER = Path("data") / RUN               # where the run's data folder is (or will be downloaded)
RESULTS = Path("results") / RUN
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. The run in AMBER

Reads the run with **your** key: in AMBER open *API & assistants*, make a read-only key; in Colab add it as the
secret `AMBER_KEY` (key icon on the left), in Jupyter put it in `~/.amber_key`. If there is no key the notebook
goes on with the data folder alone.

In [ ]:
try:
    amber = Amber(AMBER_URL)
    run = amber.run(RUN)
    print("AMBER:", run["code"], "|", run.get("title"), "| status:", run.get("status"))
    print("  start", run.get("actual_start"), " end", run.get("actual_end"))
except AmberError as e:
    run = None
    print("Not read from AMBER:", e)
    print("Going on with the data folder only.")

## 2. The data folder

On the station PC AMBER makes a folder per run with `AMBER_run.json` in it (run, experiment, samples). The station
program writes its files next to it. Here, if the folder is not on this computer, the **demo** folder is downloaded
from GitHub. With real data: upload the run's folder (Colab: folder icon on the left) and set `FOLDER`.

In [ ]:
DEMO = "https://raw.githubusercontent.com/dfme-lab/amber-notebooks/main/stations/fiaxell/demo_data/" + RUN + "/"

if not (FOLDER / "AMBER_run.json").exists():
    names = urllib.request.urlopen(DEMO + "MANIFEST.txt").read().decode().split()
    for name in names:
        (FOLDER / name).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(DEMO + name, FOLDER / name)
    print(f"Demo folder downloaded: {len(names)} files")

info = load_run_json(FOLDER)
r = info["run"]
print(r["code"], "·", r["title"])
print("station:", r["station"]["name"], "| operator:", r["operator"])
if info.get("experiment"):
    e = info["experiment"]
    print("experiment:", e["code"], "·", e["title"])
    print("  question:", e["question"])
for s in info["samples"]:
    print("sample:", s["code"], "·", s.get("title") or "", "·", s.get("composition") or "")
AREA = float(r.get("params", {}).get("active_area_cm2") or 1.0)
print("active area:", AREA, "cm2")
for p in sorted(FOLDER.rglob("*.csv")):
    print("  ", p.relative_to(FOLDER))

## 3. What happened during the run

Furnace temperature and the fuel and air flows; the dashed lines are the I-V / EIS measurements.

In [ ]:
cond = pd.read_csv(FOLDER / f"{RUN}_conditions.csv", parse_dates=["time_utc"])
temp = pd.read_csv(FOLDER / f"{RUN}_temperature.csv", parse_dates=["timestamp"])
flow = pd.read_csv(FOLDER / f"{RUN}_flow.csv", parse_dates=["timestamp"])
t0 = temp["timestamp"].iloc[0]
hours = lambda t: (t - t0).dt.total_seconds() / 3600

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours(temp["timestamp"]), temp["pv_C"], label="furnace PV")
ax.plot(hours(temp["timestamp"]), temp["setpoint_C"], "--", lw=1, label="setpoint")
ax.set_xlabel("time (h)"); ax.set_ylabel("temperature (°C)")
ax2 = ax.twinx()
for ch, c in [("H2", "tab:red"), ("N2-1", "tab:gray"), ("Air", "tab:green")]:
    ax2.plot(hours(flow["timestamp"]), flow[f"{ch}_mass_flow"], color=c, lw=1, label=ch)
ax2.set_ylabel("flow (sccm)")
for t in cond.loc[cond["event"] == "ocv", "time_utc"]:
    ax.axvline((t - t0).total_seconds() / 3600, color="k", ls=":", lw=1)
fig.legend(loc="upper right", ncol=5, fontsize=8)
ax.set_title(f"{RUN} – run overview")
fig.tight_layout(); fig.savefig(RESULTS / "overview.png", dpi=150); plt.show()

## 4. OCV check

The station writes the theoretical (Nernst) OCV and the first measured voltage. More than ~50 mV below theory points to a leak, a crack or wrong gas.

In [ ]:
ocv = cond[cond["event"] == "ocv"][["step", "T_C", "x_H2O", "OCV_theory_V", "OCV_measured_V", "OCV_deviation_mV"]]
ocv = ocv.assign(ok=ocv["OCV_deviation_mV"].abs() <= 50)
ocv.reset_index(drop=True)

## 5. I-V curves

Current density *j* = I / area, power density *P* = *j*·*V*. ASR from the slope between 0.1 and 0.5 A/cm².

In [ ]:
def t_of(path):  # temperature from the file name, e.g. ..._IV_800C.csv
    return int(path.stem.split("_")[-1].rstrip("C"))

iv_files = sorted((FOLDER / RUN).glob("*_IV_*C.csv"), key=t_of, reverse=True)
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
rows = []
for f in iv_files:
    T = t_of(f); d = pd.read_csv(f)
    j = d["current_A"] / AREA; V = d["potential_V"]; P = j * V
    a.plot(j, V, label=f"{T} °C"); b.plot(j, P, label=f"{T} °C")
    m = (j >= 0.1) & (j <= 0.5)
    asr_iv = -np.polyfit(j[m], V[m], 1)[0]
    rows.append({"T_C": T, "OCV_V": round(V.iloc[0], 4), "P_max_W_cm2": round(P.max(), 3),
                 "j_at_Pmax_A_cm2": round(j[P.idxmax()], 2), "V_at_0.5A_cm2": round(np.interp(0.5, j, V), 3),
                 "ASR_IV_ohm_cm2": round(asr_iv, 3)})
a.set_xlabel("j (A/cm²)"); a.set_ylabel("V (V)"); a.legend(); a.set_title("I-V")
b.set_xlabel("j (A/cm²)"); b.set_ylabel("P (W/cm²)"); b.legend(); b.set_title("power density")
fig.tight_layout(); fig.savefig(RESULTS / "iv.png", dpi=150); plt.show()
iv = pd.DataFrame(rows); iv

## 6. Impedance (EIS at OCV)

Nyquist plot per area. Ohmic resistance = the high-frequency crossing of the real axis; polarisation resistance = low-frequency end minus ohmic.

In [ ]:
eis_files = sorted((FOLDER / RUN).glob("*_EIS_*C.csv"), key=t_of, reverse=True)
fig, ax = plt.subplots(figsize=(7, 4))
erows = []
for f in eis_files:
    T = t_of(f); d = pd.read_csv(f).sort_values("freq_Hz", ascending=False)
    zr = d["Z_re_ohm"] * AREA; zi = -d["Z_im_ohm"] * AREA
    ax.plot(zr, zi, "o-", ms=3, label=f"{T} °C")
    k = np.where((zi.values[:-1] <= 0) & (zi.values[1:] > 0))[0]   # inductive -> capacitive
    if len(k):  # interpolate the real part where -Z'' crosses zero
        i = k[0]; f = -zi.values[i] / (zi.values[i + 1] - zi.values[i])
        r_ohm = zr.values[i] + f * (zr.values[i + 1] - zr.values[i])
    else:
        r_ohm = zr.values[0]
    r_tot = zr.values[-1]
    erows.append({"T_C": T, "R_ohm_ohm_cm2": round(r_ohm, 4), "R_pol_ohm_cm2": round(r_tot - r_ohm, 4),
                  "ASR_EIS_ohm_cm2": round(r_tot, 4)})
ax.axhline(0, color="k", lw=0.5); ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("Z' (Ω cm²)"); ax.set_ylabel("−Z'' (Ω cm²)"); ax.legend(); ax.set_title("Nyquist, OCV")
fig.tight_layout(); fig.savefig(RESULTS / "eis.png", dpi=150); plt.show()
eis = pd.DataFrame(erows); eis

## 7. Arrhenius plot

ln(1/R) against 1000/T; the slope gives the activation energy (eV).

In [ ]:
kB = 8.617e-5  # eV/K
x = 1000 / (eis["T_C"] + 273.15)
fig, ax = plt.subplots(figsize=(6, 4))
ea = {}
for col, lab in [("R_ohm_ohm_cm2", "ohmic"), ("R_pol_ohm_cm2", "polarisation")]:
    y = np.log(1 / eis[col])
    k = np.polyfit(x, y, 1)[0]
    ea[lab] = round(-k * 1000 * kB, 2)
    ax.plot(x, y, "o-", label=f"{lab}: Ea = {ea[lab]} eV")
ax.set_xlabel("1000/T (1/K)"); ax.set_ylabel("ln(1/R)  (R in Ω cm²)"); ax.legend(); ax.set_title("Arrhenius")
fig.tight_layout(); fig.savefig(RESULTS / "arrhenius.png", dpi=150); plt.show()
ea

## 8. Result for the experiment

Copy this into the experiment in AMBER (results are not written back automatically). The figures are in `results/`.

In [ ]:
res = iv.merge(eis, on="T_C").merge(ocv[["T_C", "OCV_deviation_mV"]], on="T_C")
cols = ["T_C", "OCV_V", "OCV_deviation_mV", "P_max_W_cm2", "ASR_IV_ohm_cm2", "R_ohm_ohm_cm2", "R_pol_ohm_cm2"]
try:
    text = res[cols].to_markdown(index=False)
except ImportError:  # tabulate not installed
    text = res[cols].to_string(index=False)
exp = (info.get("experiment") or {}).get("code", "")
summary = (f"Run {RUN} ({', '.join(s['code'] for s in info['samples'])}), experiment {exp}.\n\n{text}\n\n"
           f"Activation energy: ohmic {ea['ohmic']} eV, polarisation {ea['polarisation']} eV.")
(RESULTS / "summary.md").write_text(summary, encoding="utf-8")
print(summary)